In [1]:
import pandas as pd

# Load data
df = pd.read_csv("forcast_demand_for_trolley.csv")



In [2]:
df.head()

,historyId,trolleyId,geozoneId,geolayerId,date,time
0,1,1,11,7,2026-03-04,13:35:04
1,3,1,11,7,2026-03-04,13:37:14
2,8,1,11,7,2026-03-04,13:40:18
3,16,1,11,7,2026-03-04,13:42:16
4,20,1,11,7,2026-03-04,13:44:23


In [6]:
# Combine date + time
df['datetime'] = pd.to_datetime(
    df['date'] + ' ' + df['time'],
    format='%Y-%m-%d %H:%M:%S'
)

In [8]:
df.head()

,historyId,trolleyId,geozoneId,geolayerId,date,time,datetime
0,1,1,11,7,2026-03-04,13:35:04,2026-03-04 13:35:04
1,3,1,11,7,2026-03-04,13:37:14,2026-03-04 13:37:14
2,8,1,11,7,2026-03-04,13:40:18,2026-03-04 13:40:18
3,16,1,11,7,2026-03-04,13:42:16,2026-03-04 13:42:16
4,20,1,11,7,2026-03-04,13:44:23,2026-03-04 13:44:23


In [9]:
# Sort
df = df.sort_values(['trolleyId', 'datetime']).reset_index(drop=True)

df.head()

,historyId,trolleyId,geozoneId,geolayerId,date,time,datetime
0,1,1,11,7,2026-03-04,13:35:04,2026-03-04 13:35:04
1,3,1,11,7,2026-03-04,13:37:14,2026-03-04 13:37:14
2,8,1,11,7,2026-03-04,13:40:18,2026-03-04 13:40:18
3,16,1,11,7,2026-03-04,13:42:16,2026-03-04 13:42:16
4,20,1,11,7,2026-03-04,13:44:23,2026-03-04 13:44:23


# STEP 2: Create "movement change" flag

In [11]:
# Detect geozone change
df['prev_geozone'] = df.groupby('trolleyId')['geozoneId'].shift(1)

df['zone_changed'] = (df['geozoneId'] != df['prev_geozone']).astype(int)

# First row per trolley must be start
df['zone_changed'] = df['zone_changed'].fillna(1)

In [12]:
df.head()

,historyId,trolleyId,geozoneId,geolayerId,date,time,datetime,prev_geozone,zone_changed
0,1,1,11,7,2026-03-04,13:35:04,2026-03-04 13:35:04,NaN,1
1,3,1,11,7,2026-03-04,13:37:14,2026-03-04 13:37:14,11.0,0
2,8,1,11,7,2026-03-04,13:40:18,2026-03-04 13:40:18,11.0,0
3,16,1,11,7,2026-03-04,13:42:16,2026-03-04 13:42:16,11.0,0
4,20,1,11,7,2026-03-04,13:44:23,2026-03-04 13:44:23,11.0,0


# STEP 3: Assign segment IDs

In [13]:
df['segment_id'] = df.groupby('trolleyId')['zone_changed'].cumsum()

In [14]:
df.head()

,historyId,trolleyId,geozoneId,geolayerId,date,time,datetime,prev_geozone,zone_changed,segment_id
0,1,1,11,7,2026-03-04,13:35:04,2026-03-04 13:35:04,NaN,1,1
1,3,1,11,7,2026-03-04,13:37:14,2026-03-04 13:37:14,11.0,0,1
2,8,1,11,7,2026-03-04,13:40:18,2026-03-04 13:40:18,11.0,0,1
3,16,1,11,7,2026-03-04,13:42:16,2026-03-04 13:42:16,11.0,0,1
4,20,1,11,7,2026-03-04,13:44:23,2026-03-04 13:44:23,11.0,0,1


# STEP 4: Create interval dataset

In [ ]:
interval_df = df.groupby(['trolleyId', 'segment_id', 'geozoneId']).agg(
    start_time=('datetime', 'min'),
    end_time=('datetime', 'max')
).reset_index()

# Duration
interval_df['duration_min'] = (
    (interval_df['end_time'] - interval_df['start_time'])
    .dt.total_seconds() / 60
)



,trolleyId,segment_id,geozoneId,start_time,end_time,duration_min
0,1,1,11,2026-03-04 13:35:04,2026-03-04 13:51:17,16.216667
1,1,2,0,2026-03-04 13:57:33,2026-03-04 13:57:33,0.000000
2,1,3,2,2026-03-04 14:01:24,2026-03-04 14:13:10,11.766667
3,1,4,0,2026-03-04 14:22:53,2026-03-04 14:22:53,0.000000
4,1,5,3,2026-03-04 14:29:35,2026-03-04 14:40:16,10.683333


In [18]:
interval_df.tail(50)

,trolleyId,segment_id,geozoneId,start_time,end_time,duration_min
21342,500,9,1,2026-05-03 17:13:22,2026-05-03 17:13:22,0.000000
21343,500,10,16,2026-05-03 17:28:06,2026-05-03 23:02:39,334.550000
21344,500,11,1,2026-05-03 23:17:50,2026-05-03 23:32:40,14.833333
21345,500,12,0,2026-05-04 02:23:33,2026-05-04 02:53:23,29.833333
21346,500,13,6,2026-05-04 03:41:52,2026-05-04 03:41:52,0.000000
21347,500,14,17,2026-05-04 03:57:07,2026-05-04 03:57:07,0.000000
21348,500,15,16,2026-05-04 04:12:07,2026-05-10 15:32:56,9320.816667
21349,500,16,1,2026-05-10 16:27:59,2026-05-10 16:27:59,0.000000
21350,500,17,18,2026-05-10 16:43:52,2026-05-10 23:27:27,403.583333
21351,500,18,20,2026-05-10 23:42:18,2026-05-11 00:12:08,29.833333


In [19]:
interval_df.to_csv("duration_min.csv",index=False)

In [20]:
interval_df['hour'] = interval_df['start_time'].dt.hour
interval_df['day'] = interval_df['start_time'].dt.day
interval_df['weekday'] = interval_df['start_time'].dt.weekday
interval_df['is_weekend'] = interval_df['weekday'].isin([5,6]).astype(int)

In [21]:
interval_df.head()

,trolleyId,segment_id,geozoneId,start_time,end_time,duration_min,hour,day,weekday,is_weekend
0,1,1,11,2026-03-04 13:35:04,2026-03-04 13:51:17,16.216667,13,4,2,0
1,1,2,0,2026-03-04 13:57:33,2026-03-04 13:57:33,0.000000,13,4,2,0
2,1,3,2,2026-03-04 14:01:24,2026-03-04 14:13:10,11.766667,14,4,2,0
3,1,4,0,2026-03-04 14:22:53,2026-03-04 14:22:53,0.000000,14,4,2,0
4,1,5,3,2026-03-04 14:29:35,2026-03-04 14:40:16,10.683333,14,4,2,0


In [22]:
# Count trolleys per zone per hour
demand_df = interval_df.groupby(['geozoneId', 'hour']).agg(
    trolley_count=('trolleyId', 'nunique'),
    avg_duration=('duration_min', 'mean')
).reset_index()



In [24]:
demand_df.head(50)

,geozoneId,hour,trolley_count,avg_duration
0,0,0,93,40.458683
1,0,1,104,35.911640
2,0,2,201,69.944611
3,0,3,118,82.604555
4,0,4,92,30.149377
5,0,5,135,29.862828
6,0,6,148,21.931658
7,0,7,163,30.239528
8,0,8,143,29.218866
9,0,9,120,27.324717


In [25]:
demand_df = demand_df.sort_values(['geozoneId', 'hour'])

# Lag features
demand_df['lag_1'] = demand_df.groupby('geozoneId')['trolley_count'].shift(1)
demand_df['lag_2'] = demand_df.groupby('geozoneId')['trolley_count'].shift(2)

# Rolling mean
demand_df['rolling_mean'] = demand_df.groupby('geozoneId')['trolley_count']\
    .transform(lambda x: x.rolling(3).mean())

In [26]:
demand_df.head()

,geozoneId,hour,trolley_count,avg_duration,lag_1,lag_2,rolling_mean
0,0,0,93,40.458683,NaN,NaN,NaN
1,0,1,104,35.911640,93.0,NaN,NaN
2,0,2,201,69.944611,104.0,93.0,132.666667
3,0,3,118,82.604555,201.0,104.0,141.000000
4,0,4,92,30.149377,118.0,201.0,137.000000


In [27]:
final_df = demand_df.dropna()

X = final_df[['geozoneId', 'hour', 'lag_1', 'lag_2', 'rolling_mean']]
y = final_df['trolley_count']